# Modelado — Trufi App Cochabamba
## CRISP-DM Fase 4 · Técnicas evaluadas bajo un mismo protocolo — iteración 2

**Pregunta:** ¿Cómo estimar el número esperado de consultas de ruta de Trufi App por celda H3 a partir de la
población, la ubicación y el contexto territorial de cada celda, mediante técnicas geoespaciales de ciencia de datos?

El protocolo (catálogo, variables, grilla, umbral de sobredispersión y regla de adopción) se declaró y se versionó
**antes** de esta ejecución (D-201 a D-210); resumen en `EXPLANATIONS.md` §6.

**Catálogo.** Líneas base (reglas simples, sin variables explicativas): B0 tasa global, B0.5 tasa por municipio,
B0.7 tasa por anillo de distancia, B1 tasa de la vecindad H3. Modelos estadísticos: M1 GLM Poisson y M2 Binomial
Negativa, ambos con offset `log(population)`. Machine learning: M3 HistGradientBoosting con pérdida Poisson.

| Tarea | Contenido | Salida |
|---|---|---|
| M1 | Cargar y separar prueba | `particion.csv` |
| M2 | Pliegues espaciales `GroupKFold(5)` por `block_id` | `folds.csv`, `figuras/pliegues.png` |
| M3 | Sobredispersión de M1 | `sobredispersion.csv` |
| M4 | Validación cruzada espacial de todo el catálogo | `cv_espacial.csv`, `resumen_cv.csv`, `oof_predicciones.parquet` |
| M5 | Ajuste anidado de M3 | `m3_grilla.csv` |
| M6 | Validación aleatoria y optimismo | `cv_aleatoria.csv`, `optimismo_aleatorio.csv` |
| M7 | Especificaciones alternativas | `especificaciones.csv` |
| M8 | Regla de adopción en código | `regla_adopcion_pasos.csv`, `adopcion.json` |
| M9 | Sensibilidad de B1 al tamaño de vecindad (descriptiva) | `b1_sensibilidad_k.csv` |

In [1]:
import sys
from pathlib import Path


def _encontrar_raiz(inicio: Path) -> Path:
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "pyproject.toml").exists():
            return carpeta
    raise RuntimeError("No se encontró pyproject.toml")


RAIZ = _encontrar_raiz(Path.cwd())
SRC_DIR = RAIZ / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [2]:
from trufi_ds.notebook_setup import *  # noqa: F403
from trufi_ds import modeling as md
from trufi_ds.preparation import cell_polygons

import json
import geopandas as gpd
from sklearn.model_selection import GroupKFold, KFold

SEMILLA = 42
np.random.seed(SEMILLA)
SUB = "03_modelado"
crear_directorios(SUB, f"{SUB}/figuras")
REPORTE = PROJECT_ROOT / "reports" / SUB
PROCESSED = PROJECT_ROOT / "data" / "processed"
CATALOGO = {m: {} for m in md.COMPLEXITY}  # B0, B0.5, B0.7, B1, M1, M2, M3
COLORES = {"B0": "#9E9E9E", "B0.5": "#BCAAA4", "B0.7": "#A1887F", "B1": "#795548", "M1": "#1976D2", "M2": "#388E3C", "M3": "#E65100"}
METRICAS = ["poisson_deviance", "d2", "mae", "rmse", "calibration", "spearman"]

## M1 · Cargar datos y separar prueba

Regla R1: de `test_blocks.csv` se lee solo la lista de `block_id` para descartar esas filas de inmediato. Ninguna
métrica, figura ni ajuste de este notebook usa filas de prueba.

In [3]:
tabla = pl.read_parquet(PROCESSED / "tabla_minable.parquet").filter(pl.col("in_model"))
bloques_prueba = pl.read_csv(PROJECT_ROOT / "reports/02_preparacion/test_blocks.csv")["block_id"]
es_prueba = pl.col("block_id").is_in(bloques_prueba.implode())

particion = (tabla.with_columns(pl.when(es_prueba).then(pl.lit("prueba")).otherwise(pl.lit("entrenamiento")).alias("conjunto"))
             .group_by("conjunto").agg(pl.col("block_id").n_unique().alias("bloques"), pl.len().alias("celdas"),
                                       pl.col("query_count").sum().alias("consultas"))
             .sort("conjunto"))
particion.write_csv(REPORTE / "particion.csv")

datos = tabla.filter(~es_prueba).sort("h3_cell").to_pandas()
del tabla  # a partir de aquí solo existen filas de entrenamiento
assert not set(datos["block_id"]) & set(bloques_prueba.to_list()), "Solapamiento de bloques"
print(f"Entrenamiento: {len(datos):,} celdas, {datos['block_id'].nunique()} bloques, {datos['query_count'].sum():,} consultas")
particion

Entrenamiento: 1,078 celdas, 41 bloques, 1,696,323 consultas


conjunto,bloques,celdas,consultas
str,u32,u32,i64
"""entrenamiento""",41,1078,1696323
"""prueba""",12,303,228013


## M2 · Pliegues espaciales

`GroupKFold(5)` con `groups = block_id` (D-206): cada bloque H3 res 6 queda entero en un solo pliegue.

In [4]:
pliegues_esp = list(GroupKFold(5).split(datos, groups=datos["block_id"]))
fold_de = np.empty(len(datos), dtype=int)
for f, (_, va) in enumerate(pliegues_esp):
    fold_de[va] = f
folds = pd.DataFrame({"h3_cell": datos["h3_cell"], "block_id": datos["block_id"], "fold": fold_de})
folds.to_csv(REPORTE / "folds.csv", index=False)
resumen_folds = (folds.assign(q=datos["query_count"]).groupby("fold")
                 .agg(bloques=("block_id", "nunique"), celdas=("h3_cell", "size"), consultas=("q", "sum")))
resumen_folds.to_csv(REPORTE / "folds_resumen.csv")

gdf = gpd.GeoDataFrame(folds, geometry=cell_polygons(folds["h3_cell"].tolist()), crs="EPSG:4326")
fig, ax = plt.subplots(figsize=(11, 8))
gdf.plot(ax=ax, column=gdf["fold"].astype(str), categorical=True, cmap="Set2", legend=True, edgecolor="none",
         legend_kwds={"title": "pliegue", "loc": "upper right"})
ax.set_title(f"Pliegues espaciales GroupKFold(5) por bloque H3 res 6 — N={len(datos):,} celdas de entrenamiento")
ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud"); fig.tight_layout()
guardar_figura(fig, "pliegues", SUB); plt.close(fig)
resumen_folds

,bloques,celdas,consultas
fold,,,
0,8,216,69782
1,8,216,555297
2,8,216,956383
3,8,215,44152
4,9,215,70709


## M3 · Sobredispersión (D-204)

χ² de Pearson / gl residuales de M1 en cada pliegue de entrenamiento. Umbral declarado: 1,5.

In [5]:
sobre = []
for f, (tr, _) in enumerate(pliegues_esp):
    entreno = datos.iloc[tr]
    m1 = md.M1().fit(entreno)
    sobre.append({"fold": f, "pearson_chi2_df": md.pearson_dispersion(m1, entreno), "n_train": len(tr)})
sobredispersion = pd.DataFrame(sobre)
phi_media = sobredispersion["pearson_chi2_df"].mean()
REFERENCIA_GLM = "M2" if phi_media > 1.5 else "M1"
sobredispersion.to_csv(REPORTE / "sobredispersion.csv", index=False)
print(f"φ medio = {phi_media:,.1f} → referencia interpretable: {REFERENCIA_GLM}")
sobredispersion

φ medio = 45,748,347.9 → referencia interpretable: M2


,fold,pearson_chi2_df,n_train
0,0,9.528457e+05,862
1,1,2.115263e+08,862
2,2,1.364372e+07,862
3,3,2.536017e+06,863
4,4,8.290596e+04,863


## M4–M5 · Validación cruzada espacial del catálogo (con ajuste anidado de M3)

Cada técnica se ajusta en el pliegue de entrenamiento y se evalúa en el de validación. M3 elige su configuración
con `GroupKFold(3)` interno (M5). Se guardan las predicciones fuera de pliegue.

In [6]:
cv_esp, oof_esp, busqueda_esp = md.cross_validate(datos, CATALOGO, pliegues_esp)
cv_esp.to_csv(REPORTE / "cv_espacial.csv", index=False)
oof_esp.to_parquet(REPORTE / "oof_predicciones.parquet", index=False)
m3_grilla = pd.DataFrame(busqueda_esp)
m3_grilla.to_csv(REPORTE / "m3_grilla.csv", index=False)
cv_esp[["model", "fold", *METRICAS, "warnings"]].round(4)

,model,fold,poisson_deviance,d2,mae,rmse,calibration,spearman,warnings
0,B0,0,2237.1790,-0.6095,1402.8668,2067.2238,5.2190,0.9075,
1,B0.5,0,3709.5072,-1.6687,2239.2115,3817.9458,7.7132,0.8655,
2,B0.7,0,3225.4685,-1.3205,1978.6698,3043.9956,7.0904,0.9078,
3,B1,0,526.7005,0.6211,433.3041,1264.8995,2.0374,0.8850,
4,M1,0,801.9381,0.4231,648.2156,1722.1473,2.8974,0.8910,
5,M2,0,1538.3047,-0.1067,1036.8741,2307.4686,3.8189,0.9052,
6,M3,0,1056.1747,0.2402,813.9920,3360.9144,3.2036,0.9024,
7,B0,1,4854.1005,0.5127,2798.6966,7640.1772,0.9964,0.9241,
8,B0.5,1,5065.4384,0.4915,3509.1804,7593.1834,1.6280,0.8491,
9,B0.7,1,23159.7736,-1.3248,3380.4717,9298.3422,0.4484,0.7933,


In [7]:
def resumir(cv: pd.DataFrame) -> pd.DataFrame:
    dev = cv.pivot(index="fold", columns="model", values="poisson_deviance")
    mejor = dev.idxmin(axis=1)
    filas = []
    for m in [c for c in md.COMPLEXITY if c in dev.columns]:
        g = cv[cv["model"] == m]
        fila = {"model": m}
        for met in METRICAS:
            fila[f"{met}_mean"] = g[met].mean()
            fila[f"{met}_std"] = g[met].std(ddof=1)
        fila["folds_best_overall"] = int((mejor == m).sum())
        fila["folds_won_vs_B0"] = int((dev[m] < dev["B0"]).sum()) if m != "B0" else np.nan
        fila["folds_won_vs_B1"] = int((dev[m] < dev["B1"]).sum()) if m not in ("B0", "B1") else np.nan
        filas.append(fila)
    return pd.DataFrame(filas)

resumen_cv = resumir(cv_esp)
resumen_cv.to_csv(REPORTE / "resumen_cv.csv", index=False)
resumen_cv.round(4)

,model,poisson_deviance_mean,poisson_deviance_std,d2_mean,d2_std,mae_mean,mae_std,rmse_mean,rmse_std,calibration_mean,calibration_std,spearman_mean,spearman_std,folds_best_overall,folds_won_vs_B0,folds_won_vs_B1
0,B0,5234.5614,5897.5844,-0.0876,0.4713,2087.7256,1319.8109,6024.9619,6566.1044,3.1848,2.5104,0.8689,0.0513,0,NaN,NaN
1,B0.5,4557.7212,4819.7694,0.0224,0.9468,2022.4769,1697.2159,6156.9374,6303.6151,2.1516,3.1540,0.7942,0.0623,0,3.0,0.0
2,B0.7,9990.2791,11381.7855,-0.4252,0.8393,2126.1550,1674.8469,6591.9752,6795.0855,2.2773,2.8676,0.8131,0.0562,0,2.0,0.0
3,B1,1699.4262,1546.3748,0.6445,0.1205,1316.8963,1402.6235,5010.0458,4987.5150,1.1744,0.7757,0.8460,0.0399,2,5.0,NaN
4,M1,3654.9468,4320.5243,0.4774,0.0937,2038.2410,2337.3776,6651.7795,7205.7367,1.5898,1.3084,0.8582,0.0595,1,4.0,1.0
5,M2,4727.4322,6207.1183,0.3521,0.2907,1702.0673,1697.1247,6055.5425,6732.1414,1.2979,1.4424,0.8672,0.0274,1,4.0,2.0
6,M3,3055.1831,3974.3666,0.4912,0.1681,1521.1430,1477.2033,5747.8295,5982.2061,1.4304,1.1772,0.8755,0.0462,1,5.0,1.0


**Métricas por anillo.** Se agrupan las predicciones fuera de pliegue por `distance_ring` para que el centro no
domine la lectura (riesgo 3).

In [8]:
anillo = dict(zip(datos["h3_cell"], datos["distance_ring"]))
por_anillo = []
for (m, a), g in oof_esp.assign(distance_ring=oof_esp["h3_cell"].map(anillo)).groupby(["model", "distance_ring"]):
    por_anillo.append({"model": m, "distance_ring": a, **md.metrics(g["y_true"], g["y_pred"])})
cv_por_anillo = pd.DataFrame(por_anillo)
cv_por_anillo.to_csv(REPORTE / "cv_por_anillo.csv", index=False)
cv_por_anillo.pivot(index="model", columns="distance_ring", values="poisson_deviance").round(2)

distance_ring,A1,A2,A3,A4
model,,,,
B0,10789.61,4333.96,1924.00,793.72
B0.5,10707.80,3201.49,828.00,32.28
B0.7,15382.57,24188.60,854.34,10.89
B1,2327.98,3941.28,643.22,17.16
M1,6805.75,5918.44,750.59,14.21
M2,10728.15,4339.70,679.96,11.58
M3,6655.28,2766.00,826.75,25.92


In [9]:
dev = cv_esp.pivot(index="fold", columns="model", values="poisson_deviance")[md.COMPLEXITY]
fig, ax = plt.subplots(figsize=(8, 5))
for f in dev.index:
    ax.plot(range(len(dev.columns)), dev.loc[f], color="#BDBDBD", lw=1, zorder=1)
for i, m in enumerate(dev.columns):
    ax.scatter([i] * len(dev), dev[m], color=COLORES[m], s=40, zorder=2)
    ax.errorbar(i + 0.15, dev[m].mean(), yerr=dev[m].std(ddof=1), fmt="D", color="black", ms=5, capsize=4, zorder=3)
ax.set_xticks(range(len(dev.columns)), dev.columns); ax.set_yscale("log")
ax.set_ylabel("Devianza Poisson media (validación, log)")
ax.set_title(f"Devianza por pliegue espacial — N={len(datos):,} celdas, 5 pliegues\n(gris: un pliegue; ◆: media ± DE)")
fig.tight_layout(); guardar_figura(fig, "devianza_por_pliegue", SUB); plt.close(fig)

## M6 · Validación aleatoria y optimismo

Mismo universo y mismas técnicas con `KFold(5, shuffle=True, random_state=42)`. El optimismo es ΔD² = D² aleatorio
− D² espacial: cuánto sobrestima una validación que mezcla celdas vecinas entre entrenamiento y validación.

In [10]:
pliegues_ale = list(KFold(5, shuffle=True, random_state=SEMILLA).split(datos))
catalogo_ale = {**CATALOGO, "M3": {"inner_random": True}}
cv_ale, oof_ale, _ = md.cross_validate(datos, catalogo_ale, pliegues_ale)
cv_ale.to_csv(REPORTE / "cv_aleatoria.csv", index=False)

optimismo = (cv_esp.groupby("model")[["d2", "poisson_deviance"]].mean().add_suffix("_espacial")
             .join(cv_ale.groupby("model")[["d2", "poisson_deviance"]].mean().add_suffix("_aleatoria"))
             .reindex(md.COMPLEXITY))
optimismo["delta_d2"] = optimismo["d2_aleatoria"] - optimismo["d2_espacial"]
optimismo["ratio_devianza_espacial_aleatoria"] = optimismo["poisson_deviance_espacial"] / optimismo["poisson_deviance_aleatoria"]
optimismo.reset_index().to_csv(REPORTE / "optimismo_aleatorio.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(optimismo))
ax.bar(x - 0.2, optimismo["d2_espacial"], 0.4, label="Validación espacial (bloques)", color="#1976D2")
ax.bar(x + 0.2, optimismo["d2_aleatoria"], 0.4, label="Validación aleatoria", color="#90CAF9")
for i, d in enumerate(optimismo["delta_d2"]):
    ax.annotate(f"Δ={d:+.2f}", (i, max(optimismo["d2_espacial"].iloc[i], optimismo["d2_aleatoria"].iloc[i])),
                ha="center", va="bottom", fontsize=9, xytext=(0, 3), textcoords="offset points")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x, optimismo.index); ax.set_ylabel("D² medio (5 pliegues)")
ax.set_title(f"Optimismo de la validación aleatoria — N={len(datos):,} celdas")
ax.legend(); fig.tight_layout(); guardar_figura(fig, "optimismo_aleatorio", SUB); plt.close(fig)
optimismo.round(4)

,d2_espacial,poisson_deviance_espacial,d2_aleatoria,poisson_deviance_aleatoria,delta_d2,ratio_devianza_espacial_aleatoria
model,,,,,,
B0,-0.0876,5234.5614,0.4809,4525.1900,0.5685,1.1568
B0.5,0.0224,4557.7212,0.5746,3675.3793,0.5522,1.2401
B0.7,-0.4252,9990.2791,0.5184,4263.1498,0.9437,2.3434
B1,0.6445,1699.4262,0.9063,830.6235,0.2618,2.0460
M1,0.4774,3654.9468,0.7243,2558.6152,0.2470,1.4285
M2,0.3521,4727.4322,0.5638,3953.7484,0.2117,1.1957
M3,0.4912,3055.1831,0.7596,2066.2421,0.2684,1.4786


## M7 · Especificaciones alternativas (D-207)

(a) `log(population)` como covariable libre en M1 y M2; (b) objetivo `user_count` con todo el catálogo. Solo en
validación espacial, con los mismos pliegues.

In [11]:
cv_libre, _, _ = md.cross_validate(datos, {"M1_free": {"free_offset": True}, "M2_free": {"free_offset": True}}, pliegues_esp)
cv_usuarios, _, _ = md.cross_validate(datos, CATALOGO, pliegues_esp, target="user_count")

filas_esp = []
for base in ["M1", "M2"]:
    d_off = cv_esp[cv_esp["model"] == base].set_index("fold")["poisson_deviance"]
    d_lib = cv_libre[cv_libre["model"] == f"{base}_free"].set_index("fold")["poisson_deviance"]
    filas_esp.append({"especificacion": f"{base} con log(population) libre", "objetivo": "query_count",
                      "deviance_mean": d_lib.mean(), "deviance_std": d_lib.std(ddof=1),
                      "d2_mean": cv_libre[cv_libre["model"] == f"{base}_free"]["d2"].mean(),
                      "deviance_referencia": d_off.mean(), "mejora_pct_vs_offset": (1 - d_lib.mean() / d_off.mean()) * 100,
                      "pliegues_ganados_vs_offset": int((d_lib < d_off).sum())})
for m, g in cv_usuarios.groupby("model"):
    filas_esp.append({"especificacion": m, "objetivo": "user_count", "deviance_mean": g["poisson_deviance"].mean(),
                      "deviance_std": g["poisson_deviance"].std(ddof=1), "d2_mean": g["d2"].mean(),
                      "deviance_referencia": np.nan, "mejora_pct_vs_offset": np.nan, "pliegues_ganados_vs_offset": np.nan})
especificaciones = pd.DataFrame(filas_esp)
especificaciones.to_csv(REPORTE / "especificaciones.csv", index=False)
coef_libre = {m: md.make(m, free_offset=True).fit(datos).result_.params["log_population"] for m in ["M1", "M2"]}
adoptada_usuarios, _ = md.adoption_rule(cv_usuarios)
print("Coeficiente libre de log(population) (ajuste con todo el entrenamiento):", {k: round(v, 3) for k, v in coef_libre.items()})
print("Técnica que adoptaría la regla con user_count:", adoptada_usuarios)
especificaciones.round(4)

Coeficiente libre de log(population) (ajuste con todo el entrenamiento): {'M1': np.float64(1.5), 'M2': np.float64(1.073)}
Técnica que adoptaría la regla con user_count: B1


,especificacion,objetivo,deviance_mean,deviance_std,d2_mean,deviance_referencia,mejora_pct_vs_offset,pliegues_ganados_vs_offset
0,M1 con log(population) libre,query_count,3651.1714,4315.6045,0.4799,3654.9468,0.1033,4.0
1,M2 con log(population) libre,query_count,4717.9187,6205.5235,0.3545,4727.4322,0.2012,5.0
2,B0,user_count,1390.4834,1480.5243,0.0343,NaN,NaN,NaN
3,B0.5,user_count,1210.1845,1212.6249,0.1376,NaN,NaN,NaN
4,B0.7,user_count,2626.7689,2948.3343,-0.2695,NaN,NaN,NaN
5,B1,user_count,474.5632,399.4563,0.6566,NaN,NaN,NaN
6,M1,user_count,923.7578,1035.8732,0.5134,NaN,NaN,NaN
7,M2,user_count,1253.3139,1563.6643,0.3928,NaN,NaN,NaN
8,M3,user_count,810.7037,1017.9489,0.5276,NaN,NaN,NaN


## M9 · Sensibilidad de B1 al tamaño de vecindad (D-210, descriptiva)

B1 busca celdas de entrenamiento alrededor de cada celda hasta reunir `min_neighbors`, con un radio máximo `k_max`.
Se prueban otras combinaciones solo para ver si el resultado depende de esa elección; **no** se usan para elegir.

In [12]:
variantes_b1 = {f"B1_n{n}_k{k}": {"min_neighbors": n, "k_max": k} for n in (1, 3, 5) for k in (3, 10)}
cv_b1, _, _ = md.cross_validate(datos, variantes_b1, pliegues_esp)
b1_sens = (cv_b1.groupby("model").agg(poisson_deviance_mean=("poisson_deviance", "mean"), poisson_deviance_std=("poisson_deviance", "std"),
                                      d2_mean=("d2", "mean"), spearman_mean=("spearman", "mean"), calibration_mean=("calibration", "mean"))
           .reset_index())
b1_sens[["min_neighbors", "k_max"]] = [[v["min_neighbors"], v["k_max"]] for v in b1_sens["model"].map(variantes_b1)]
b1_sens["configuracion_adoptada"] = (b1_sens["min_neighbors"] == 3) & (b1_sens["k_max"] == 10)
b1_sens.to_csv(REPORTE / "b1_sensibilidad_k.csv", index=False)
b1_sens.round(3)

,model,poisson_deviance_mean,poisson_deviance_std,d2_mean,spearman_mean,calibration_mean,min_neighbors,k_max,configuracion_adoptada
0,B1_n1_k10,1585.158,1489.073,0.675,0.829,1.080,1,10,False
1,B1_n1_k3,1572.286,1161.808,0.595,0.712,1.338,1,3,False
2,B1_n3_k10,1699.426,1546.375,0.645,0.846,1.174,3,10,True
3,B1_n3_k3,1692.491,1239.111,0.494,0.685,1.713,3,3,False
4,B1_n5_k10,1803.437,1595.601,0.615,0.846,1.244,5,10,False
5,B1_n5_k3,2897.816,3380.772,0.325,0.680,2.055,5,3,False


## M8 · Regla de adopción (D-205), aplicada en código

In [13]:
ADOPTADA, log_regla = md.adoption_rule(cv_esp)
log_regla.to_csv(REPORTE / "regla_adopcion_pasos.csv", index=False)

elegidas = m3_grilla[m3_grilla["chosen"]]
frecuencia = (elegidas.groupby(list(md.M3_GRID), dropna=False)
              .agg(veces=("fold", "size"), dev_interna=("inner_deviance_mean", "mean"))
              .reset_index().sort_values(["veces", "dev_interna"], ascending=[False, True]))
m3_config = {k: (None if pd.isna(v) else (int(v) if k != "learning_rate" else float(v)))
             for k, v in frecuencia.iloc[0][list(md.M3_GRID)].items()}

adopcion = {"adopted": ADOPTADA, "reference_glm": REFERENCIA_GLM, "overdispersion_phi_mean": phi_media,
            "m3_final_params": m3_config, "free_offset_coefficients": coef_libre,
            "spatial_folds": "GroupKFold(5) por block_id", "seed": SEMILLA}
(REPORTE / "adopcion.json").write_text(json.dumps(adopcion, indent=2), encoding="utf-8")

print(f"Técnica adoptada: {ADOPTADA} · referencia interpretable: {REFERENCIA_GLM}")
log_regla

Técnica adoptada: B1 · referencia interpretable: M2


,candidate,vs_best_simpler,deviance_candidate,deviance_best_simpler,improvement_pct,folds_won,passes,adopted_after_step
0,B0.5,B0,4557.721192,5234.561413,12.930218,3,False,B0
1,B0.7,B0.5,9990.279100,4557.721192,-119.194608,2,False,B0
2,B1,B0.5,1699.426188,4557.721192,62.713248,5,True,B1
3,M1,B1,3654.946847,1699.426188,-115.069467,1,False,B1
4,M2,B1,4727.432243,1699.426188,-178.178145,2,False,B1
5,M3,B1,3055.183103,1699.426188,-79.777335,1,False,B1
